In [5]:
# 1. Imports
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

from imblearn.over_sampling import SMOTE

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping


# 2. Load Dataset
df = pd.read_csv("/content/drive/MyDrive/CollegeProjects/Sid/AIOT_PROJECT2/diabetes_generated.csv")

X = df.drop("Outcome", axis=1)
y = df["Outcome"]


# 3. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


# 4. Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# 5. SMOTE (only training data)
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)


# 6. Build Neural Network
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_res.shape[1],)),
    BatchNormalization(),
    Dropout(0.3),

    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),

    Dense(32, activation='relu'),
    Dropout(0.2),

    Dense(1, activation='sigmoid')
])


# 7. Compile Model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)


# 8. Early Stopping
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)


# 9. Train Model
history = model.fit(
    X_train_res, y_train_res,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)


# 10. Evaluate Model
loss, acc = model.evaluate(X_test_scaled, y_test)

print("\n Neural Network Result ")
print("Accuracy:", acc)


# 11. Save Model + Scaler
model.save("/content/drive/MyDrive/CollegeProjects/Sid/AIOT_PROJECT2/NeuralNetwork/diabetes_nn_model.h5")
joblib.dump(scaler, "nn_scaler.pkl")

print(" Model saved as diabetes_nn_model.h5")
print(" Scaler saved as nn_scaler.pkl")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/100
220/220 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - accuracy: 0.7364 - loss: 0.5476 - val_accuracy: 0.7926 - val_loss: 0.4806
Epoch 2/100
220/220 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7666 - loss: 0.4900 - val_accuracy: 0.7955 - val_loss: 0.4436
Epoch 3/100
220/220 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7767 - loss: 0.4784 - val_accuracy: 0.8023 - val_loss: 0.4400
Epoch 4/100
220/220 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7814 - loss: 0.4628 - val_accuracy: 0.8006 - val_loss: 0.4361
Epoch 5/100
220/220 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7908 - loss: 0.4596 - val_accuracy: 0.8051 - val_loss: 0.4296
Epoch 6/100
220/220 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7864 - loss: 0.4551 - val_accuracy: 0.7983 - val_loss: 0.4318
Epoch 7/100
220/220 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7905 - loss: 0.4493 - val_accuracy: 0.8068 - val_loss: 0.4258
Epoch 8/100
220/220 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7928 - loss: 0.4447 - val_acc


 Neural Network Result 
Accuracy: 0.8156917095184326
 Model saved as diabetes_nn_model.h5
 Scaler saved as nn_scaler.pkl
